In [30]:
import os
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
import psycopg2
import pandas as pd


In [31]:
DB = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": int(os.getenv("DB_PORT", 5432)),
    "dbname": os.getenv("DB_NAME", "client_query_management"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASS", "kovilvenni"),
}
DB


{'host': 'localhost',
 'port': 5432,
 'dbname': 'client_query_management',
 'user': 'postgres',
 'password': 'kovilvenni'}

In [32]:
try:
    # Try connecting to the target DB
    conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname=DB['dbname'],
                            user=DB['user'], password=DB['password'])
    conn.close()
    print(f"Connected to database {DB['dbname']} at {DB['host']}:{DB['port']}")
except Exception as e:
    print('Could not connect to target DB — attempting to create it if possible.')
    try:
        admin_conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname='postgres',
                                      user=DB['user'], password=DB['password'])
        admin_conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cur = admin_conn.cursor()
        cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB['dbname'],))
        exists = cur.fetchone()
        if not exists:
            cur.execute(f"CREATE DATABASE {DB['dbname']}")
            print(f"Database {DB['dbname']} created.")
        else:
            print(f"Database {DB['dbname']} already exists.")
        cur.close()
        admin_conn.close()
    except Exception as e2:
        print('Failed to create database. Please create it manually or check credentials. Error:', e2)

# Now try to connect again
try:
    conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname=DB['dbname'],
                            user=DB['user'], password=DB['password'])
    conn.close()
    print('Connection successful.')
except Exception as e:
    print('Final connection attempt failed:', e)


Connected to database client_query_management at localhost:5432
Connection successful.


In [33]:
sql = """
CREATE TABLE IF NOT EXISTS users (
    id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    username VARCHAR(100) UNIQUE NOT NULL,
    hashed_password VARCHAR(255) NOT NULL,
    role TEXT CHECK (role IN ('Client', 'Support')) NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS queries (
    query_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    mail_id VARCHAR(255),
    mobile_number VARCHAR(50),
    query_heading VARCHAR(255),
    query_description TEXT,
    status TEXT CHECK (status IN ('Open', 'Closed')) DEFAULT 'Open',
    query_created_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    query_closed_time TIMESTAMP
);
"""

conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname=DB['dbname'],
                        user=DB['user'], password=DB['password'])
cur = conn.cursor()
cur.execute(sql)
conn.commit()
cur.close()
conn.close()
print("Tables created (if they did not exist).")


Tables created (if they did not exist).


In [34]:
sql = """
INSERT INTO users (username, hashed_password, role) VALUES
('arun', '1f1d5f931045a65e39114639ae0567c60cd8fa32f43ce275a73375735d350fbb', 'Support'),
('luffy', 'db6450ac95b15d9059234e6f419795c8cdc3e78a8901a1334ab49fb4b02c519e', 'Support'),
('zoro', 'eb1457569fb72d472c2a99c841e306a18a30d5274e5075beb84198c24ea86f1d', 'Support'),
('nami', '631599f283ec12b96b671c3e1538b393a98439cc0a583b671a5e87754965711c', 'Support'),
('sanji', 'af13bda5c95305094358715bceaa0261673dfbf6e2bd193ebcd7d9b902225396', 'Support'),
('ace', 'c701962105e5ce8d891d4b3548ba02a2b71acc8c00206dc23235ba22847e83b4', 'Client'),
('sabo', '6b97c549cdb7ff3ce7d0dc0425e21a90d013b1be1957d88798a5c0d455933712', 'Client'),
('yamato', 'f41486e6d352c7f440aa64b4e4bd22b7bddeeec7a8b019b065cc1d12a8b90d84', 'Client'),
('dragon', 'c20cf9879666a76500bb136af3d0cf4d15683064b3ef5bdad5ced4ca083926ba', 'Client');
"""

conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname=DB['dbname'],
                        user=DB['user'], password=DB['password'])
cur = conn.cursor()
try:
    cur.execute(sql)
    conn.commit()
    print("Inserted sample users.")
except Exception as e:
    conn.rollback()
    print("Insert users failed (possibly duplicates):", e)
cur.close()
conn.close()


Insert users failed (possibly duplicates): duplicate key value violates unique constraint "users_username_key"
DETAIL:  Key (username)=(arun) already exists.



In [35]:
sql = """
INSERT INTO queries (mail_id, mobile_number, query_heading, query_description, status, query_created_time, query_closed_time) VALUES
('arun@gmail.com','9000000012','Login Issue','Unable to log in using correct credentials','Open',NOW(),NULL),
('zoro@gmail.com','9000000014','Login Issue','Unable to login to account after password reset','Open',NOW(),NULL),
('nami@gmail.com','9000000015','Refund Delay','Refund not received even after 7 days','Open',NOW(),NULL),
('sanji@gmail.com','9000000016','Account Locked','Too many failed attempts, account locked','Open',NOW(),NULL),
('usopp@gmail.com','9000000017','Order Not Delivered','Order status shows delivered but not received','Open',NOW(),NULL),
('robin@gmail.com','9000000018','Email Not Verified','Verification email not received','Open',NOW(),NULL),
('franky@gmail.com','9000000019','Payment Declined','Card is valid but payment keeps failing','Open',NOW(),NULL),
('brook@gmail.com','9000000020','App Crash','Application crashes when clicking checkout','Open',NOW(),NULL),
('chopper@gmail.com','9000000021','Wrong Amount Charged','Extra charges applied during order placement','Open',NOW(),NULL),
('ace@gmail.com','9000000022','Slow Website','Website taking too long to load product pages','Open',NOW(),NULL),
('jinbei@gmail.com','9000000023','Multiple Attempts Charged','Charged twice for one transaction','Open',NOW(),NULL);
"""

conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname=DB['dbname'],
                        user=DB['user'], password=DB['password'])
cur = conn.cursor()
try:
    cur.execute(sql)
    conn.commit()
    print("Inserted sample queries.")
except Exception as e:
    conn.rollback()
    print("Insert queries failed (possibly duplicates):", e)
cur.close()
conn.close()


Inserted sample queries.


In [36]:
conn = psycopg2.connect(host=DB['host'], port=DB['port'], dbname=DB['dbname'],
                       user=DB['user'], password=DB['password'])

df_users = pd.read_sql('SELECT * FROM users ORDER BY id;', conn)
df_queries = pd.read_sql('SELECT * FROM queries ORDER BY query_created_time ;', conn)
conn.close()

display(df_users)
print("\n---\n")
display(df_queries)


C:\Users\arunm\AppData\Local\Temp\ipykernel_44596\1198002055.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_users = pd.read_sql('SELECT * FROM users ORDER BY id;', conn)
C:\Users\arunm\AppData\Local\Temp\ipykernel_44596\1198002055.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_queries = pd.read_sql('SELECT * FROM queries ORDER BY query_created_time ;', conn)


,id,username,hashed_password,role,created_at
0,1,arun,1f1d5f931045a65e39114639ae0567c60cd8fa32f43ce2...,Support,2025-12-06 13:20:26.429133
1,2,luffy,db6450ac95b15d9059234e6f419795c8cdc3e78a8901a1...,Support,2025-12-06 13:20:26.429133
2,3,zoro,eb1457569fb72d472c2a99c841e306a18a30d5274e5075...,Support,2025-12-06 13:20:26.429133
3,4,nami,631599f283ec12b96b671c3e1538b393a98439cc0a583b...,Support,2025-12-06 13:20:26.429133
4,5,sanji,af13bda5c95305094358715bceaa0261673dfbf6e2bd19...,Support,2025-12-06 13:20:26.429133
5,12,ace,c701962105e5ce8d891d4b3548ba02a2b71acc8c00206d...,Client,2025-12-06 13:20:26.429133
6,13,sabo,6b97c549cdb7ff3ce7d0dc0425e21a90d013b1be1957d8...,Client,2025-12-06 13:20:26.429133
7,14,yamato,f41486e6d352c7f440aa64b4e4bd22b7bddeeec7a8b019...,Client,2025-12-06 13:20:26.429133
8,15,dragon,c20cf9879666a76500bb136af3d0cf4d15683064b3ef5b...,Client,2025-12-06 13:20:26.429133
9,16,kalam,c592a19032c469868bb3bfccfa43a4f3ca8e348673dcb9...,Client,2025-12-06 16:09:18.489733



---



,query_id,mail_id,mobile_number,query_heading,query_description,status,query_created_time,query_closed_time
0,3,nami@gmail.com,9000000015,Refund Delay,Refund not received even after 7 days,Open,2025-12-05 23:10:39.807699,NaT
1,4,sanji@gmail.com,9000000016,Account Locked,"Too many failed attempts, account locked",Open,2025-12-05 23:10:39.807699,NaT
2,2,zoro@gmail.com,9000000014,Login Issue,Unable to login to account after password reset,Open,2025-12-05 23:10:39.807699,NaT
3,5,usopp@gmail.com,9000000017,Order Not Delivered,Order status shows delivered but not received,Open,2025-12-05 23:10:39.807699,NaT
4,6,robin@gmail.com,9000000018,Email Not Verified,Verification email not received,Open,2025-12-05 23:10:39.807699,NaT
...,...,...,...,...,...,...,...,...
62,62,robin@gmail.com,9000000018,Email Not Verified,Verification email not received,Open,2025-12-07 02:34:09.736332,NaT
63,63,franky@gmail.com,9000000019,Payment Declined,Card is valid but payment keeps failing,Open,2025-12-07 02:34:09.736332,NaT
64,64,brook@gmail.com,9000000020,App Crash,Application crashes when clicking checkout,Open,2025-12-07 02:34:09.736332,NaT
65,65,chopper@gmail.com,9000000021,Wrong Amount Charged,Extra charges applied during order placement,Open,2025-12-07 02:34:09.736332,NaT
